# 🏗️ Building an Evaluation Pipeline

**Day 9 — Notebook 4 of 4 | Estimated Time: 35 minutes**

---

## What You'll Learn
- How to structure a repeatable end-to-end evaluation pipeline
- Combining retrieval and generation metrics into one run
- Comparing multiple system variants systematically
- Generating an evaluation report with a comparison dashboard
- Detecting regressions when you change the system

---

## What a Production Eval Pipeline Looks Like

```
Golden Dataset (Q&A + relevance labels)
        │
        ▼
System Under Test (RAG pipeline)
  ├─ Retrieval → Precision@K, Recall@K, MRR
  └─ Generation → Faithfulness, Relevance, Correctness
        │
        ▼
Eval Report
  ├─ Per-query breakdown (diagnose failures)
  ├─ Aggregate metrics (track over time)
  └─ System comparison (A vs B)
        │
        ▼
Decision: ship, iterate, or investigate
```

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb

In [ ]:
import sys, os, json, math, time
from statistics import mean, stdev
from dataclasses import dataclass, field
import chromadb

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from pydantic import BaseModel

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Rebuild the HR Policy System

We'll use the same corpus from Notebook 3, now creating two system variants to compare.

In [ ]:
CORPUS = [
    {"id": "annual-leave",      "text": "Employees receive 25 days annual leave per year, accruing at 2.08 days per month. Leave must be requested 5 days in advance via the HR portal."},
    {"id": "sick-leave",        "text": "Employees are entitled to 10 days paid sick leave per year. A doctor's note is required for absences over 3 consecutive days. Sick leave does not carry over."},
    {"id": "parental-leave",    "text": "Primary caregivers receive 18 weeks fully paid parental leave. Secondary caregivers receive 4 weeks. Employees need 6 months tenure to qualify."},
    {"id": "remote-work",       "text": "Employees may work remotely up to 3 days per week. All employees must be in the office on Wednesdays. International remote work is limited to 20 days per year."},
    {"id": "office-equipment",  "text": "A £500 annual allowance is provided for home office equipment. Purchases must be approved via the procurement portal. Receipts are required."},
    {"id": "expense-meals",     "text": "Meal expenses during business travel are capped at £50 per person per day. Alcohol is not reimbursable. Receipts required for claims over £10."},
    {"id": "expense-travel",    "text": "Economy class is required for flights under 6 hours. Business class may be approved for longer flights. Hotel accommodation is capped at £200 in London."},
    {"id": "salary-review",     "text": "Salary reviews occur every April. Increases are based on performance ratings from the annual review cycle. The typical range is 2-8% for strong performers."},
    {"id": "bonus-policy",      "text": "Discretionary bonuses are paid in December based on company and individual performance. Target bonus is 10% of annual salary for most roles."},
    {"id": "health-benefits",   "text": "All employees receive private health insurance. Dental and vision plans are optional add-ons. Benefits start from the first day of employment."},
    {"id": "pension",           "text": "The company contributes 6% of salary to the pension scheme. Employee contributions start at 3%. The combined 9% minimum meets auto-enrolment requirements."},
    {"id": "training-budget",   "text": "Each employee has a £1,500 annual learning budget for courses, books, and conferences. Manager approval is required for single items over £200."},
]

GOLDEN_DATASET = [
    {"id": "q1", "query": "How much annual leave do I get?",
     "ground_truth": "25 days per year, accruing at 2.08 days per month.",
     "relevant_ids": ["annual-leave"]},
    {"id": "q2", "query": "What leave is available for new parents?",
     "ground_truth": "Primary caregivers: 18 weeks. Secondary caregivers: 4 weeks. Requires 6 months tenure.",
     "relevant_ids": ["parental-leave", "annual-leave"]},
    {"id": "q3", "query": "Can I claim hotel and meal costs on a business trip?",
     "ground_truth": "Hotel: up to £200 in London. Meals: up to £50/person/day. Receipts needed over £10.",
     "relevant_ids": ["expense-travel", "expense-meals"]},
    {"id": "q4", "query": "What are the remote work rules for working abroad?",
     "ground_truth": "International remote work is limited to 20 days per year.",
     "relevant_ids": ["remote-work"]},
    {"id": "q5", "query": "How does the company contribute to my pension?",
     "ground_truth": "Company contributes 6%. Employee minimum is 3%.",
     "relevant_ids": ["pension"]},
    {"id": "q6", "query": "What health benefits do I get?",
     "ground_truth": "Private health insurance from day one. Optional dental and vision add-ons.",
     "relevant_ids": ["health-benefits"]},
]

# Build the index
chroma = chromadb.Client()
col = chroma.create_collection("hr", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in CORPUS]
result = client.models.embed_content(
    model=EMBEDDING_MODEL, contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
col.add(
    ids=[d["id"] for d in CORPUS],
    documents=texts,
    embeddings=[e.values for e in result.embeddings],
)
print(f"✅ Indexed {col.count()} documents, {len(GOLDEN_DATASET)} eval queries")

---

## 2. Define System Variants

We'll compare three increasingly sophisticated configurations.

In [ ]:
def embed_query(query: str) -> list[float]:
    return client.models.embed_content(
        model=EMBEDDING_MODEL, contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values


def retrieve_docs(query: str, top_k: int) -> tuple[list[str], list[str]]:
    """Returns (doc_ids, doc_texts) in rank order."""
    q_vec = embed_query(query)
    results = col.query(
        query_embeddings=[q_vec], n_results=top_k,
        include=["documents", "ids"],
    )
    return results["ids"][0], results["documents"][0]


# ─── System A: Vanilla LLM (no retrieval) ─────────────────────────────
def system_a(query: str) -> tuple[str, list[str]]:
    """Vanilla LLM — no context, no retrieval."""
    response = client.models.generate_content(
        model=GEN_MODEL, contents=query,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip(), []   # no retrieved IDs


# ─── System B: Basic RAG (top_k=3, no compression) ─────────────────────
def system_b(query: str) -> tuple[str, list[str]]:
    """Basic RAG — dense retrieval, top-3, no postprocessing."""
    doc_ids, doc_texts = retrieve_docs(query, top_k=3)
    context = "\n\n".join(doc_texts)
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"Answer using ONLY the HR policy context below.\n\n<context>\n{context}\n</context>\n\nQuestion: {query}",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip(), doc_ids


# ─── System C: RAG with explicit citation instructions ──────────────────
SYSTEM_C_PROMPT = """You are an HR policy assistant. Answer questions concisely using ONLY the 
provided policy context. Include specific numbers and details. If the answer is not in the 
context, say: "This information is not available in the provided policy documents."""

def system_c(query: str) -> tuple[str, list[str]]:
    """RAG with structured system prompt and top-5 retrieval."""
    doc_ids, doc_texts = retrieve_docs(query, top_k=5)
    context = "\n\n".join(f"Policy: {t}" for t in doc_texts)
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=[
            types.Content(role="user", parts=[
                types.Part(text=f"{SYSTEM_C_PROMPT}\n\n<context>\n{context}\n</context>\n\nQuestion: {query}")
            ])
        ],
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip(), doc_ids


SYSTEMS = {
    "system_a": {"fn": system_a, "name": "Vanilla LLM",         "retrieval_k": 0},
    "system_b": {"fn": system_b, "name": "Basic RAG (k=3)",     "retrieval_k": 3},
    "system_c": {"fn": system_c, "name": "Structured RAG (k=5)", "retrieval_k": 5},
}

print("Systems defined:")
for key, s in SYSTEMS.items():
    print(f"  {key}: {s['name']}")

---

## 3. The Evaluation Engine

In [ ]:
# ─── Metric functions (from Notebook 3) ────────────────────────────────
def precision_at_k(retrieved, relevant, k):
    hits = sum(1 for d in retrieved[:k] if d in relevant)
    return hits / k if k > 0 else 0.0

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 1.0
    hits = sum(1 for d in retrieved[:k] if d in relevant)
    return hits / len(relevant)

def reciprocal_rank(retrieved, relevant):
    for i, d in enumerate(retrieved):
        if d in relevant:
            return 1.0 / (i + 1)
    return 0.0


# ─── LLM judge (from Notebook 2) ───────────────────────────────────────
class JudgeScore(BaseModel):
    faithfulness: int    # 1–5
    relevance: int       # 1–5
    correctness: int     # 1–5
    reasoning: str


def judge(question: str, context: str, answer: str, ground_truth: str) -> JudgeScore:
    """Score all three generation dimensions in a single LLM call."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Evaluate the answer on three dimensions. Score each 1–5.

Faithfulness (1=answer contradicts context, 5=fully grounded in context):
Relevance (1=doesn't address the question, 5=directly answers it):
Correctness (1=wrong vs ground truth, 5=matches ground truth exactly):

Context: {context if context else 'No context provided (vanilla LLM)'}

Question: {question}
Ground truth: {ground_truth}
Answer: {answer}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=JudgeScore,
            temperature=0.0,
        ),
    )
    return JudgeScore.model_validate_json(response.text)


def run_evaluation(
    system_fn,
    system_name: str,
    golden_dataset: list[dict],
    retrieval_k: int = 0,
) -> dict:
    """
    Run a complete evaluation of one system against the golden dataset.
    Returns a report dict with per-query results and aggregate metrics.
    """
    results = []

    for item in golden_dataset:
        query = item["query"]
        ground_truth = item["ground_truth"]
        relevant_ids = item["relevant_ids"]

        # Run the system
        t0 = time.time()
        answer, retrieved_ids = system_fn(query)
        latency = time.time() - t0

        # Retrieval metrics (only meaningful if system retrieves)
        if retrieval_k > 0:
            prec = precision_at_k(retrieved_ids, relevant_ids, retrieval_k)
            rec = recall_at_k(retrieved_ids, relevant_ids, retrieval_k)
            rr = reciprocal_rank(retrieved_ids, relevant_ids)
        else:
            prec = rec = rr = None

        # Get context text for judge
        id_to_text = {d["id"]: d["text"] for d in CORPUS}
        context = "\n\n".join(id_to_text[d] for d in retrieved_ids if d in id_to_text)

        # Generation quality (judge)
        gen_scores = judge(query, context, answer, ground_truth)

        results.append({
            "query_id": item["id"],
            "query": query,
            "answer": answer,
            "retrieved_ids": retrieved_ids,
            "latency": latency,
            "precision": prec,
            "recall": rec,
            "rr": rr,
            "faithfulness": gen_scores.faithfulness,
            "relevance": gen_scores.relevance,
            "correctness": gen_scores.correctness,
        })
        print(f"  [{item['id']}] done ({latency:.1f}s)")

    # Aggregate metrics
    def safe_mean(values):
        valid = [v for v in values if v is not None]
        return mean(valid) if valid else None

    return {
        "system": system_name,
        "results": results,
        "aggregate": {
            "mean_precision": safe_mean([r["precision"] for r in results]),
            "mean_recall":    safe_mean([r["recall"] for r in results]),
            "mrr":            safe_mean([r["rr"] for r in results]),
            "mean_faithfulness": safe_mean([r["faithfulness"] for r in results]),
            "mean_relevance":    safe_mean([r["relevance"] for r in results]),
            "mean_correctness":  safe_mean([r["correctness"] for r in results]),
            "mean_latency":      safe_mean([r["latency"] for r in results]),
        },
    }


print("Evaluation engine ready.")

In [ ]:
# Run evaluation for all three systems
all_reports = {}

for sys_key, sys_config in SYSTEMS.items():
    print(f"\nEvaluating {sys_config['name']}...")
    report = run_evaluation(
        system_fn=sys_config["fn"],
        system_name=sys_config["name"],
        golden_dataset=GOLDEN_DATASET,
        retrieval_k=sys_config["retrieval_k"],
    )
    all_reports[sys_key] = report

print("\n✅ All systems evaluated!")

---

## 4. Comparison Dashboard

In [ ]:
def print_dashboard(all_reports: dict):
    """Print a side-by-side comparison of all evaluated systems."""
    systems = list(all_reports.keys())
    names = [all_reports[s]["system"] for s in systems]

    def fmt(val):
        if val is None:
            return "  N/A "
        return f" {val:.3f}"

    print("\n" + "=" * 70)
    print("EVALUATION DASHBOARD")
    print("=" * 70)

    # Header
    col_w = 20
    print(f"{'Metric':<22}" + "".join(f"{n[:col_w]:>{col_w}}" for n in names))
    print("-" * 70)

    metrics = [
        ("RETRIEVAL", None),
        ("  Precision@K",       "mean_precision"),
        ("  Recall@K",          "mean_recall"),
        ("  MRR",               "mrr"),
        ("GENERATION", None),
        ("  Faithfulness",      "mean_faithfulness"),
        ("  Relevance",         "mean_relevance"),
        ("  Correctness",       "mean_correctness"),
        ("PERFORMANCE", None),
        ("  Avg Latency (s)",   "mean_latency"),
    ]

    for label, key in metrics:
        if key is None:
            print(f"\n{label}")
            continue
        vals = [all_reports[s]["aggregate"][key] for s in systems]
        # Highlight best value
        valid_vals = [(i, v) for i, v in enumerate(vals) if v is not None]
        if valid_vals:
            if key == "mean_latency":
                best_i = min(valid_vals, key=lambda x: x[1])[0]
            else:
                best_i = max(valid_vals, key=lambda x: x[1])[0]
        else:
            best_i = -1

        row = f"{label:<22}"
        for i, v in enumerate(vals):
            cell = fmt(v)
            if i == best_i and v is not None:
                cell = f" ★{v:.3f}"  # mark the winner
            row += f"{cell:>{col_w}}"
        print(row)

    print("\n  ★ = best score for that metric")
    print("=" * 70)


print_dashboard(all_reports)

---

## 5. Per-Query Failure Analysis

In [ ]:
def find_failures(report: dict, faithfulness_threshold: int = 3, correctness_threshold: int = 3):
    """Identify queries where the system performed poorly."""
    failures = []
    for r in report["results"]:
        issues = []
        if r["faithfulness"] is not None and r["faithfulness"] < faithfulness_threshold:
            issues.append(f"low faithfulness ({r['faithfulness']}/5)")
        if r["correctness"] is not None and r["correctness"] < correctness_threshold:
            issues.append(f"low correctness ({r['correctness']}/5)")
        if r["recall"] is not None and r["recall"] == 0.0:
            issues.append("retrieval miss (relevant doc not found)")
        if issues:
            failures.append({**r, "issues": issues})
    return failures


# Analyse failures per system
for sys_key in SYSTEMS:
    failures = find_failures(all_reports[sys_key])
    name = all_reports[sys_key]["system"]
    print(f"\n{name} — {len(failures)} failures:")
    if not failures:
        print("  ✅ No failures detected")
    for f in failures:
        print(f"  [{f['query_id']}] {f['query'][:50]}")
        print(f"    Issues: {'; '.join(f['issues'])}")
        print(f"    Answer: {f['answer'][:100]}...")

---

## 6. Regression Detection

When you change your system, you want to know whether you improved or regressed.

In [ ]:
def compare_systems(
    baseline_report: dict,
    new_report: dict,
    regression_threshold: float = 0.05,  # Flag if metric drops by more than this
) -> dict:
    """
    Compare a new system against a baseline.
    Returns deltas and flags regressions.
    """
    metrics = ["mean_faithfulness", "mean_relevance", "mean_correctness",
               "mean_precision", "mean_recall", "mrr"]

    comparison = {}
    has_regression = False

    for metric in metrics:
        baseline_val = baseline_report["aggregate"][metric]
        new_val = new_report["aggregate"][metric]

        if baseline_val is None or new_val is None:
            comparison[metric] = {"baseline": baseline_val, "new": new_val, "delta": None, "regression": False}
            continue

        delta = new_val - baseline_val
        is_regression = delta < -regression_threshold
        if is_regression:
            has_regression = True

        comparison[metric] = {
            "baseline": baseline_val,
            "new": new_val,
            "delta": delta,
            "regression": is_regression,
        }

    return {"comparison": comparison, "has_regression": has_regression}


def print_comparison(result: dict, baseline_name: str, new_name: str):
    print(f"\nCOMPARISON: {baseline_name} → {new_name}")
    print(f"{'Metric':<25} {'Baseline':>10} {'New':>8} {'Delta':>8}")
    print("-" * 55)
    for metric, vals in result["comparison"].items():
        if vals["delta"] is None:
            print(f"  {metric:<23}     N/A       N/A      N/A")
            continue
        flag = " 🔴" if vals["regression"] else (" 🟢" if vals["delta"] > 0.02 else "   ")
        print(
            f"  {metric:<23}   {vals['baseline']:.3f}    {vals['new']:.3f}   "
            f"{vals['delta']:+.3f}{flag}"
        )

    if result["has_regression"]:
        print("\n  🔴 REGRESSION DETECTED — review before shipping")
    else:
        print("\n  ✅ No regressions detected")


# System B → System C comparison
comparison_b_c = compare_systems(all_reports["system_b"], all_reports["system_c"])
print_comparison(comparison_b_c, "Basic RAG (B)", "Structured RAG (C)")

# System A → System B comparison
comparison_a_b = compare_systems(all_reports["system_a"], all_reports["system_b"])
print_comparison(comparison_a_b, "Vanilla LLM (A)", "Basic RAG (B)")

---

## 7. Save the Evaluation Report

In [ ]:
import datetime

def save_eval_report(all_reports: dict, path: str = "/tmp/eval_report.json"):
    """Persist the evaluation results for future comparison."""
    report_data = {
        "timestamp": datetime.datetime.utcnow().isoformat(),
        "systems": {},
    }
    for sys_key, report in all_reports.items():
        report_data["systems"][sys_key] = {
            "name": report["system"],
            "aggregate": report["aggregate"],
            # Per-query results (omit full answer text to keep size manageable)
            "per_query": [
                {
                    "query_id": r["query_id"],
                    "faithfulness": r["faithfulness"],
                    "relevance": r["relevance"],
                    "correctness": r["correctness"],
                    "precision": r["precision"],
                    "recall": r["recall"],
                }
                for r in report["results"]
            ],
        }

    with open(path, "w") as f:
        json.dump(report_data, f, indent=2)

    print(f"✅ Report saved to {path}")
    return path


report_path = save_eval_report(all_reports)

# Show a summary snippet
with open(report_path) as f:
    saved = json.load(f)

print(f"\nReport timestamp: {saved['timestamp']}")
print("Systems included:")
for sys_key, sys_data in saved["systems"].items():
    print(f"  {sys_key}: {sys_data['name']}")
    agg = sys_data["aggregate"]
    print(f"    correctness={agg.get('mean_correctness', 'N/A'):.3f}, faithfulness={agg.get('mean_faithfulness', 'N/A'):.3f}")

---

## 🏋️ Exercise 1: Add a System D

Implement and evaluate a new system variant — either better or deliberately worse — and see what the pipeline catches.

In [ ]:
# Exercise 1: Build and evaluate System D
# Choose one of these variants to implement:
#
# Option A (should improve): High-temperature retrieval + query rewriting
#   Use Gemini to rewrite the query before embedding, then retrieve
#   Hypothesis: rewritten queries should improve Recall
#
# Option B (should degrade): Low-K retrieval (k=1) with no system prompt
#   Hypothesis: lower recall, lower faithfulness
#
# Option C (creative): Add a brief "reflection" step where the model checks its own answer
#   Hypothesis: higher correctness, higher latency

def system_d(query: str) -> tuple[str, list[str]]:
    # TODO: Implement your chosen variant
    pass


# TODO: Add to SYSTEMS and run_evaluation()
# Then compare against System B using compare_systems()

# system_d_report = run_evaluation(system_d, "System D", GOLDEN_DATASET, retrieval_k=3)
# comparison = compare_systems(all_reports["system_b"], system_d_report)
# print_comparison(comparison, "Basic RAG (B)", "System D")

---

## 🏋️ Exercise 2: Category-Level Breakdown

Aggregate scores can hide category-specific failures. Break the report down by query category.

In [ ]:
# Exercise 2: Category-level analysis
# The golden dataset queries span multiple HR topics:
#   leave, expenses, remote work, pension, benefits
#
# TODO:
# 1. Add a "category" field to each item in GOLDEN_DATASET:
#    e.g. "leave", "expenses", "remote", "compensation", "benefits"
#
# 2. Modify the evaluation or post-process the results to group scores by category
#
# 3. Build a per-category report for System C:
#    print: category | Precision | Recall | Faithfulness | Correctness
#
# 4. Answer: are there categories where System C consistently under-performs?
#    If so, what might explain it?

GOLDEN_WITH_CATEGORIES = [
    {**q, "category": "TODO: assign category"}
    for q in GOLDEN_DATASET
]

# TODO: Group and print per-category results from all_reports["system_c"]

---

## Key Takeaways

| Component | Purpose |
|-----------|---------|
| Golden dataset | Fixed ground truth — version-controlled, never used for tuning |
| System variants | Compare changes systematically rather than relying on intuition |
| Retrieval metrics | Precision@K, Recall@K, MRR — separate from generation quality |
| Generation metrics | Faithfulness, Relevance, Correctness — LLM-judged at temperature=0 |
| Dashboard | Side-by-side view of all metrics across all systems |
| Regression detection | Flag when a new change drops any metric below threshold |
| Saved reports | JSON export → compare runs over time, detect drift |

---

## Day 9 Complete! 🎉

You've learned:
- ✅ Why evaluation matters and how to build golden datasets (Notebook 1)
- ✅ LLM-as-judge for faithfulness, relevance, and correctness (Notebook 2)
- ✅ Retrieval metrics: Precision@K, Recall@K, MRR, NDCG (Notebook 3)
- ✅ End-to-end evaluation pipeline with system comparison and regression detection (Notebook 4)

**Next:** Day 10 — Agents, Function Calling & Tool Integration

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [RAGAS](https://docs.ragas.io/) | Library | Production-grade RAG evaluation framework |
| [DeepEval](https://github.com/confident-ai/deepeval) | Library | Open-source LLM evaluation framework |
| [Confident AI](https://www.confident-ai.com/) | Blog | Evaluation best practices for LLM apps |